# Inspect Verification Database
View verification reports, sources, price data, and buy recommendations stored in `data/transcripts.db`.

In [ ]:
import json
import os
import sqlite3
import sys
from pathlib import Path

# Ensure working directory is the project root, not notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

from src.database.transcript_db import TranscriptDatabase
import pandas as pd

pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_columns", None)

db = TranscriptDatabase()

## Verifications

In [ ]:
with sqlite3.connect(db.db_path) as conn:
    verifications = pd.read_sql_query("SELECT * FROM verifications", conn)

print(f"{len(verifications)} verification(s) in database")
verifications[["video_id", "model_used", "verified_at"]]

## Verifications joined with Transcript metadata

In [ ]:
transcripts = db.load()
merged = verifications.merge(
    transcripts[["video_id", "channel", "title", "url"]], on="video_id"
)
merged[["channel", "title", "model_used", "verified_at", "url"]]

## Verification Reports

In [ ]:
from IPython.display import Markdown

for _, row in merged.iterrows():
    display(Markdown(f"### [{row['title']}]({row['url']})"))
    display(
        Markdown(
            f"*Channel: {row['channel']} | Model: {row['model_used']} | {row['verified_at']}*"
        )
    )
    display(Markdown(row["verification_report"]))
    display(Markdown("---"))

## Sources

In [ ]:
for _, row in merged.iterrows():
    sources = json.loads(row["sources"])
    display(Markdown(f"### [{row['title']}]({row['url']})"))
    if sources:
        display(Markdown("\n".join(f"- {s}" for s in sources)))
    else:
        display(Markdown("*No sources found.*"))
    display(Markdown("---"))

## Price Data

In [ ]:
for _, row in merged.iterrows():
    price_data = json.loads(row["price_data"])
    display(Markdown(f"### [{row['title']}]({row['url']})"))
    if price_data:
        price_df = pd.DataFrame(
            [{"card": card, "data": info} for card, info in price_data.items()]
        )
        display(price_df)
    else:
        display(Markdown("*No price data found.*"))
    display(Markdown("---"))

## Buy Recommendations

In [ ]:
summaries = db.load_summaries()
refined = summaries[summaries["buy_recommendations"].notna()].merge(
    transcripts[["video_id", "channel", "title", "url"]], on="video_id"
)

print(f"{len(refined)} buy recommendation(s) in database")

for _, row in refined.iterrows():
    display(Markdown(f"### [{row['title']}]({row['url']})"))
    display(
        Markdown(
            f"*Channel: {row['channel']} | Generated at: {row['recommendations_at']}*"
        )
    )
    display(Markdown(row["buy_recommendations"]))
    display(Markdown("---"))

## Pending (unverified summaries)

In [ ]:
pending = db.get_unverified_summaries()
print(f"{len(pending)} summary/summaries not yet verified")
if not pending.empty:
    display(pending[["video_id", "channel", "title"]])